In [5]:
import torch
import torch.nn as nn
import numpy as np

X = torch.tensor([
    [22.0, 0.0],
    [25.0, 0.5],
    [28.0, 0.1],
    [38.0, 0.0],
    [23.0, 3.5],
    [40.0,2.0]
], dtype=torch.float32)

y = torch.tensor([
    [0.0],
    [0.0],
    [0.0],
    [1.0],
    [1.0],
    [1.0]
], dtype=torch.float32)




In [6]:
class TinyMLModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2,2)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(2,1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = TinyMLModel()

In [8]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)

for epoch in range(500):
    optimizer.zero_grad()
    predictions = model(X)
    loss = criterion(predictions, y)
    loss.backward()
    optimizer.step()


print(f"Training Complete. Final Loss:{loss.item():.4f}")


Training Complete. Final Loss:0.6931


In [10]:
# Extract parameters from PyTorch tensors into standard NumPy arrays
w1 = model.fc1.weight.detach().numpy().T  # Transpose to shape [INPUT_DIM][HIDDEN_DIM]
b1 = model.fc1.bias.detach().numpy()
w2 = model.fc2.weight.detach().numpy().T  # Transpose to shape [HIDDEN_DIM][OUTPUT_DIM]
b2 = model.fc2.bias.detach().numpy()

print(w1)
print(b1)
print(w2)
print(b2)

# Format and print raw parameter values as C array syntax
print("// --- Paste these arrays directly into your nRF52840 C code ---\n")
print(f"static const float W1[2][2] = {{\n  {{{w1[0][0]:.4f}f, {w1[0][1]:.4f}f}},\n  {{{w1[1][0]:.4f}f, {w1[1][1]:.4f}f}}\n}};")
print(f"static const float B1[2] = {{{b1[0]:.4f}f, {b1[1]:.4f}f}};\n")
print(f"static const float W2[2][1] = {{\n  {{{w2[0][0]:.4f}f}},\n  {{{w2[1][0]:.4f}f}}\n}};")
print(f"static const float B2[1] = {{{b2[0]:.4f}f}};")

[[-0.29863074 -0.6499998 ]
 [ 0.00963306 -0.5332039 ]]
[0.18109676 0.66955316]
[[ 0.06646185]
 [-0.47465813]]
[-7.951746e-08]
// --- Paste these arrays directly into your nRF52840 C code ---

static const float W1[2][2] = {
  {-0.2986f, -0.6500f},
  {0.0096f, -0.5332f}
};
static const float B1[2] = {0.1811f, 0.6696f};

static const float W2[2][1] = {
  {0.0665f},
  {-0.4747f}
};
static const float B2[1] = {-0.0000f};
